In [ ]:
import os
import sys
notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, '../..')) # Adjust as needed
if project_root not in sys.path:
    sys.path.append(project_root) # add notebook to sys.path

In [ ]:
import torch

In [ ]:
device = torch.device('cpu') if not torch.accelerator.is_available() else torch.accelerator.current_accelerator()
print(device)

# Hyperparameters

In [ ]:
batch_size = 128
resize = (224, 224)
num_epochs = 20
lr = 0.001

# Load dataset

In [ ]:
from utils.data import CIFAR10
from utils.train import train_model, val_stats


def get_optimizer(model):
    return torch.optim.Adam(model.parameters(), lr=lr)

def fit(model, num_epochs=num_epochs):
    dataset = CIFAR10(batch_size=batch_size, resize=resize, device=device)
    train_dl, val_dl = dataset.get_dataloaders()
    optimizer = get_optimizer(model)
    return train_model(model, train_dl, val_dl, optimizer, num_epochs)

# Create model

In [ ]:
from torch import nn
from utils.models import Module

In [ ]:
def vgg_block(num_convs, out_channels):
    layers = []
    for _ in range(num_convs):
        layers.append(nn.LazyConv2d(out_channels, kernel_size=3, padding=1))
        layers.append(nn.ReLU())
    layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
    return nn.Sequential(*layers)

In [ ]:
class VGG(Module):
    def __init__(self, arch, dropout=0.5, num_classes=10):
        super().__init__()
        self.dropout = dropout
        self.num_classes = num_classes
        conv_blocks = []
        for (num_convs, out_channels) in arch:
            conv_blocks.append(vgg_block(num_convs, out_channels))
        self.net = nn.Sequential(
            *conv_blocks, nn.Flatten(),
            nn.Linear(25088, 4096), nn.ReLU(), nn.Dropout(self.dropout),
            nn.LazyLinear(4096), nn.ReLU(), nn.Dropout(self.dropout),
            nn.LazyLinear(num_classes)
        )
    
    def forward(self, X):
        return self.net(X)

In [ ]:
arch = ((1, 64), (1, 128), (2, 256), (2, 512), (2, 512))
model = VGG(arch).to(device)
X = torch.randn(batch_size, 3, 224, 224).to(device)

In [ ]:
with torch.no_grad():
    print(model(X).shape)

# How many parameters in model?

First VGG Block: (3, 224, 224) -> Conv -> (64, 224, 224) -> MaxPool -> (64, 112, 112)
* Params: 3 * 64 * 3 * 3 + 64 = 1792

Second VGG Block: (64, 112, 112) -> Conv -> (128, 112, 112) -> MaxPool -> (128, 56, 56)
* Params: 64 * 128 * 3 * 3 + 128 = 73,856

Third VGG Block: (128, 56, 56) -> Conv -> (256, 56, 56) -> Conv -> (256, 56, 56) -> MaxPool -> (256, 28, 28)
* Params: 128 * 256 * 3 * 3 + 256 + 256 * 256 * 3 * 3 + 256 = 885,248

Fourth VGG Block: (256, 28, 28) -> Conv -> (512, 28, 28) -> Conv -> (512, 28, 28) -> MaxPool -> (512, 14, 14)
* Params: 256 * 512 * 3 * 3 + 512 + 512 * 512 * 3 * 3 + 512 = 3,539,968

Fifth VGG Block:  (512, 14, 14) -> Conv -> (512, 14, 14) -> Conv -> (512, 14, 14) -> MaxPool -> (512, 7, 7)
* Params: 512 * 512 * 3 * 3 + 512 + 512 * 512 * 3 * 3 + 512 = 4,719,616

Linear 1: (25088) -> (4096)
* Params: 25,088 * 4096 + 4096 = 102,764,544

Linear 2: (4096) -> (4096)
* Params: 4096 * 4096 + 4096 = 16,781,312

Linear 3: (4096) -> (10)
* Params: 4096 * 10 + 10 = 40970

In [ ]:
# hand-calculated number of params

calculated_num_params = 1792 + 73856 + 885248 + 3539968 + 4719616 + 102764544 + 16781312 + 40970
num_params = sum(p.numel() for p in model.parameters())

print(calculated_num_params, num_params)

print(f"Parameter memory: {num_params * 4 / 1e6:.0f} MB")

# Train model

In [ ]:
fit(model);

In [ ]:
_, val_dl = CIFAR10(batch_size=batch_size, resize=resize, device=device).get_dataloaders()
val_loss, val_accuracy = val_stats(model, val_dl, device=device)

In [ ]:
print(f"Validation loss: {1.00:.2f}, accuracy: {72:.2f}%")

# How much memory usage during training?

Our dataset is 60,000 images which are reshaped to (224, 224, 3)

Suppose we have a batch size of 128. 

In [ ]:
print(f"Max memory allocated: {torch.cuda.max_memory_allocated()/1e9:.3f} GB")